<a href="https://colab.research.google.com/github/Kauustubbh/Skin-Lesion/blob/main/notebooks/03_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive, userdata

# 1. Mount Google Drive to access test_split CSV and model weights
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/SkinLesionClassifier'

# 2. Authenticate Kaggle API using Colab Secrets
try:
    token = userdata.get('Kaggle_api_key')
except:
    token = userdata.get('KAGGLE_API_TOKEN')

os.environ['KAGGLE_API_TOKEN'] = token

# 3. Download and unpack dataset to Colab local disk
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 --unzip -p data/
print("Dataset downloaded and unzipped successfully.")

In [ ]:
import glob
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Evaluation running on device: {device}")

## Load Data Contract and Resolve Test Paths

In [ ]:
# 1. Load the shared split file
split_csv_path = os.path.join(PROJECT_DIR, 'train_val_test_split.csv')
assert os.path.exists(split_csv_path), f"Split file missing at {split_csv_path}"
df = pd.read_csv(split_csv_path)

# 2. Filter strictly for the held-out test split
test_df = df[df['split'] == 'test'].reset_index(drop=True)
print(f"Total test set samples: {len(test_df)}")

# 3. Map image IDs to file paths across both downloaded folders
img_paths = glob.glob('data/HAM10000_images_part_1/*.jpg') + glob.glob('data/HAM10000_images_part_2/*.jpg')
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in img_paths}
test_df['file_path'] = test_df['image_id'].map(img_lookup)

# Verify no missing file paths
assert test_df['file_path'].isnull().sum() == 0, "Error: Some test images were not located!"

# 4. Canonical class label encoding matching Notebook 02
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_DESCRIPTIONS = {
    'akiec': 'Actinic Keratosis',
    'bcc':   'Basal Cell Carcinoma',
    'bkl':   'Benign Keratosis',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevus',
    'vasc':  'Vascular Lesion'
}

label_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
idx_to_label = {idx: name for idx, name in enumerate(CLASS_NAMES)}
test_df['label'] = test_df['dx'].map(label_to_idx)

## Define Test Dataset and DataLoader

In [ ]:
# Validation/Test transform pipeline: No random augmentations
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class SkinLesionTestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['file_path']).convert('RGB')
        label = row['label']

        if self.transform:
            image = self.transform(image)

        return image, label, row['image_id'], row['file_path']

test_dataset = SkinLesionTestDataset(test_df, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print("Test DataLoader successfully initialized.")

## Reconstruct Architecture and Load Weights

In [ ]:
# 1. Instantiate identical architecture
model = resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))

# 2. Load trained weights from Drive
weights_path = os.path.join(PROJECT_DIR, 'baseline_resnet50.pth')
assert os.path.exists(weights_path), f"Checkpoint not found at {weights_path}!"

state_dict = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()
print(f"Loaded trained state dictionary from: {weights_path}")

## Collect Test Set Predictions

In [ ]:
y_true = []
y_pred = []
y_probs = []
image_ids = []
file_paths = []

softmax = nn.Softmax(dim=1)

with torch.no_grad():
    for images, labels, ids, paths in test_loader:
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        probs = softmax(outputs)
        _, preds = torch.max(probs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())
        image_ids.extend(ids)
        file_paths.extend(paths)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

print(f"Collected predictions across {len(y_pred)} unseen test samples.")

## Generate and Plot Normalized Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=[f"{c}\n({CLASS_NAMES[i]})" for i, c in enumerate(CLASS_NAMES)],
    yticklabels=[f"{CLASS_DESCRIPTIONS[c]} ({c})" for c in CLASS_NAMES],
    cbar=True,
    ax=ax
)

plt.title('Normalized Confusion Matrix — ResNet-50 (HAM10000 Test Set)', fontsize=13, pad=12)
plt.xlabel('Predicted Diagnosis', fontsize=11)
plt.ylabel('Ground Truth Biopsy Label', fontsize=11)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

# Save figure to Google Drive
cm_output_path = os.path.join(PROJECT_DIR, 'confusion_matrix.png')
plt.savefig(cm_output_path, dpi=300)
plt.show()
print(f"Confusion matrix plot saved to: {cm_output_path}")

## Output Per-Class Precision, Recall, and F1-Score

In [ ]:
# Full classification report
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=[f"{CLASS_DESCRIPTIONS[c]} ({c})" for c in CLASS_NAMES],
    output_dict=True
)

report_df = pd.DataFrame(report_dict).transpose()
print(classification_report(
    y_true,
    y_pred,
    target_names=[f"{CLASS_DESCRIPTIONS[c]} ({c})" for c in CLASS_NAMES],
    digits=4
))

# Save report metrics to Drive
report_csv_path = os.path.join(PROJECT_DIR, 'test_metrics_report.csv')
report_df.to_csv(report_csv_path)
print(f"Metrics table saved to: {report_csv_path}")

## Qualitative Error Analysis (Visualizing Misclassified Images)

In [ ]:
# Create error analysis DataFrame
results_df = pd.DataFrame({
    'image_id': image_ids,
    'file_path': file_paths,
    'true_label': y_true,
    'pred_label': y_pred,
    'true_class': [CLASS_NAMES[i] for i in y_true],
    'pred_class': [CLASS_NAMES[i] for i in y_pred],
    'confidence': [y_probs[idx][pred] for idx, pred in enumerate(y_pred)],
    'mel_true_prob': [y_probs[idx][label_to_idx['mel']] for idx in range(len(y_true))]
})

# Filter: True Melanoma misclassified as Benign Mole (nv) or Benign Keratosis (bkl)
critical_errors = results_df[
    (results_df['true_class'] == 'mel') & (results_df['pred_class'].isin(['nv', 'bkl']))
].sort_values(by='confidence', ascending=False)

print(f"Total False-Negative Melanoma errors identified: {len(critical_errors)}")

# Plot the top 4 most confident critical errors
if len(critical_errors) > 0:
    samples_to_plot = min(4, len(critical_errors))
    fig, axes = plt.subplots(1, samples_to_plot, figsize=(16, 4))
    if samples_to_plot == 1:
        axes = [axes]

    for idx, (_, row) in enumerate(critical_errors.head(samples_to_plot).iterrows()):
        img = Image.open(row['file_path'])
        axes[idx].imshow(img)
        title_str = (
            f"ID: {row['image_id']}\n"
            f"True: {CLASS_DESCRIPTIONS[row['true_class']]}\n"
            f"Pred: {CLASS_DESCRIPTIONS[row['pred_class']]}\n"
            f"Conf: {row['confidence']*100:.1f}%\n"
            f"(Mel Prob: {row['mel_true_prob']*100:.1f}%)"
        )
        axes[idx].set_title(title_str, fontsize=10, color='darkred')
        axes[idx].axis('off')

    plt.tight_layout()
    error_plot_path = os.path.join(PROJECT_DIR, 'melanoma_error_analysis.png')
    plt.savefig(error_plot_path, dpi=300)
    plt.show()
    print(f"Critical error plot saved to: {error_plot_path}")

## Deliverables Verification

In [ ]:
print("--- Evaluation Deliverables Verification ---")
print(f"1. Confusion Matrix Plot: {'EXISTS' if os.path.exists(cm_output_path) else 'MISSING'}")
print(f"2. Metrics Report CSV:   {'EXISTS' if os.path.exists(report_csv_path) else 'MISSING'}")
print(f"3. Error Analysis Plot:   {'EXISTS' if os.path.exists(error_plot_path) else 'MISSING'}")
print(f"Macro Average F1-Score: {report_dict['macro avg']['f1-score']:.4f}")
print(f"Weighted Average F1-Score: {report_dict['weighted avg']['f1-score']:.4f}")
print("Evaluation stage complete.")